In [2]:
!uv pip install spacy
!uv pip install google-generativeai
!uv pip install ratelimit

⠙ Resolving dependencies...                                                     

Resolved 42 packages in 934ms                                        
⠙ Preparing packages... (0/19)                                                  
⠙ Preparing packages... (0/19)-------------     0 B/120.50 KiB          
⠙ Preparing packages... (0/19)------------- 14.91 KiB/120.50 KiB        
⠙ Preparing packages... (0/19)------------- 30.91 KiB/120.50 KiB        
⠙ Preparing packages... (0/19)------------- 46.91 KiB/120.50 KiB        
⠙ Preparing packages... (0/19)------------- 62.91 KiB/120.50 KiB        
⠙ Preparing packages... (0/19)---------- 78.91 KiB/120.50 KiB        
⠙ Preparing packages... (0/19)--------- 94.91 KiB/120.50 KiB        
⠙ Preparing packages... (0/19)--------- 110.91 KiB/120.50 KiB       
⠙ Preparing packages... (0/19)--------- 120.50 KiB/120.50 KiB       
⠙ Preparing packages... (0/19)--------- 120.50 KiB/120.50 KiB       
wasabi               ------------------------------     0 B/27.23 KiB
⠙ Preparing packages... (0/19)--------- 120.50 KiB/120.50 KiB       

In [3]:
!uv pip install PyPDF2

⠙ Resolving dependencies...                                                     

Resolved 1 package in 198ms                                          
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)--------------     0 B/227.12 KiB          
⠙ Preparing packages... (0/1)-------------- 16.00 KiB/227.12 KiB        
⠙ Preparing packages... (0/1)-------------- 32.00 KiB/227.12 KiB        
⠙ Preparing packages... (0/1)-------------- 48.00 KiB/227.12 KiB        
⠙ Preparing packages... (0/1)-------------- 60.33 KiB/227.12 KiB        
⠙ Preparing packages... (0/1)-------------- 76.33 KiB/227.12 KiB        
⠙ Preparing packages... (0/1)-------------- 92.33 KiB/227.12 KiB        
⠙ Preparing packages... (0/1)-------------- 108.33 KiB/227.12 KiB       
⠙ Preparing packages... (0/1)m------------- 124.33 KiB/227.12 KiB       
⠙ Preparing packages... (0/1)----------- 140.33 KiB/227.12 KiB       
⠙ Preparing packages... (0/1)---------- 156.33 KiB/227.12 KiB       
⠙ Preparing packages... (0/1)---------- 172.33 KiB/22

In [4]:
!uv pip install ml-dtypes
!uv pip install codecarbon


Resolved 2 packages in 208ms                                         
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)--------------     0 B/4.71 MiB            
⠙ Preparing packages... (0/1)-------------- 16.00 KiB/4.71 MiB          
⠙ Preparing packages... (0/1)-------------- 32.00 KiB/4.71 MiB          
⠙ Preparing packages... (0/1)-------------- 48.00 KiB/4.71 MiB          
⠙ Preparing packages... (0/1)-------------- 64.00 KiB/4.71 MiB          
⠙ Preparing packages... (0/1)-------------- 80.00 KiB/4.71 MiB          
⠙ Preparing packages... (0/1)-------------- 96.00 KiB/4.71 MiB          
⠙ Preparing packages... (0/1)-------------- 112.00 KiB/4.71 MiB         
⠙ Preparing packages... (0/1)-------------- 128.00 KiB/4.71 MiB         
⠙ Preparing packages... (0/1)-------------- 144.00 KiB/4.71 MiB         
⠙ Preparing packages... (0/1)-------------- 160.00 KiB/4.71 MiB         
⠙ Preparing packages... (0/1)-------------- 17

In [7]:
from pathlib import Path
from dotenv import load_dotenv

# point to your .env file
env_path = Path(".").parent / ".env"
load_dotenv(env_path)

# now os.getenv("GOOGLE_API_KEY") will pick it up


True

In [12]:
# in your virtualenv:
!uv pip install spacy
!python -m pip install spacy


Audited 1 package in 19ms


/home/students/Leishmania/.venv/bin/python: No module named pip


In [ ]:
import pandas as pd
import PyPDF2
import fitz  # PyMuPDF for better text extraction and image handling
from pathlib import Path
import json
import re
import hashlib
from typing import List, Dict, Tuple, Optional, Set, Any
import logging
from datetime import datetime
import shutil
import os
import multiprocessing as mp
from functools import partial
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
import pickle
import time

# New imports for semantic processing
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
import spacy

# For LLM-powered enrichment (you'll need to install and configure)
import google.generativeai as genai
from google.generativeai.types import HarmCategory, HarmBlockThreshold

# --- Enhanced Configuration ---
TEXTBOOK_SOURCE_DIR = Path("data/all_leishmania_sources")
RAG_OUTPUT_DIR = Path("kaggle/working/rag_knowledge_base")
FINETUNE_OUTPUT_DIR = Path("kaggle/working/fine_tuning_data")
PROCESSED_METADATA_PATH = Path("kaggle/working/textbook_processing_metadata.csv")
CHECKPOINT_PATH = Path("kaggle/working/processing_checkpoint.json")
FILE_HASHES_PATH = Path("kaggle/working/file_hashes.json")

# New semantic processing config
SEMANTIC_CHUNKS_DIR = Path("kaggle/working/semantic_chunks")
ENRICHED_CHUNKS_DIR = Path("kaggle/working/enriched_chunks")
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"  # Lightweight but effective
LLM_MODEL_NAME = "gemini-1.5-flash"  # Fast and cost-effective

# Chunk configuration
MIN_CHUNK_SIZE = 200  # Minimum characters
MAX_CHUNK_SIZE = 800  # Maximum characters  
CHUNK_OVERLAP = 100   # Overlap between chunks
MIN_RELEVANCE_SCORE = 0.6  # Minimum relevance to keep chunk
HIGH_RELEVANCE_THRESHOLD = 0.8  # Core vs long-tail distinction

# Parallel processing
MAX_WORKERS = min(4, mp.cpu_count())
CHUNK_SIZE = 1

# Configure Gemini (you'll need to set your API key)
# genai.configure(api_key="YOUR_API_KEY_HERE")

# Enhanced logging
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(processName)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('tokenizers/punkt_tab')       # <— check for the one you’re missing
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('punkt',    quiet=True)
    nltk.download('punkt_tab',quiet=True)         # <— grab the missing tokenizer
    nltk.download('stopwords',quiet=True)


class HeadingExtractor:
    """Extract document structure and headings from PDFs"""
    
    def __init__(self):
        self.heading_patterns = [
            # Common medical heading patterns
            r'^(INTRODUCTION|CASE\s+PRESENTATION|DIAGNOSIS|TREATMENT|DISCUSSION|CONCLUSION|ABSTRACT|SUMMARY|BACKGROUND|METHODS|RESULTS)',
            r'^[0-9]+\.\s+[A-Z][A-Za-z\s]+',  # Numbered headings
            r'^[A-Z][A-Z\s]{5,}$',  # ALL CAPS headings
        ]
    
    def extract_document_structure(self, pdf_path: Path) -> Dict:
        """Extract full text with heading structure preserved"""
        try:
            doc = fitz.open(pdf_path)
            document_structure = {
                'filename': pdf_path.name,
                'sections': [],
                'full_text': "",
                'headings': [],
                'fonts': {}  # Track font information
            }
            
            current_section = {
                'heading': 'Introduction',
                'level': 1,
                'content': '',
                'page_start': 1,
                'page_end': 1
            }
            
            for page_num, page in enumerate(doc):
                # Get text with font information
                blocks = page.get_text("dict")
                
                for block in blocks["blocks"]:
                    if "lines" in block:
                        for line in block["lines"]:
                            line_text = ""
                            line_fonts = []
                            
                            for span in line["spans"]:
                                text = span["text"].strip()
                                if text:
                                    line_text += text + " "
                                    
                                    # Collect font information
                                    font_info = {
                                        'size': span['size'],
                                        'font': span['font'],
                                        'flags': span['flags']  # Bold, italic, etc.
                                    }
                                    line_fonts.append(font_info)
                            
                            line_text = line_text.strip()
                            if line_text:
                                # Determine if this is a heading
                                is_heading, heading_level = self._is_heading(line_text, line_fonts)
                                
                                if is_heading:
                                    # Save previous section
                                    if current_section['content'].strip():
                                        current_section['page_end'] = page_num
                                        document_structure['sections'].append(current_section.copy())
                                    
                                    # Start new section
                                    current_section = {
                                        'heading': line_text,
                                        'level': heading_level,
                                        'content': '',
                                        'page_start': page_num + 1,
                                        'page_end': page_num + 1
                                    }
                                    
                                    document_structure['headings'].append({
                                        'text': line_text,
                                        'level': heading_level,
                                        'page': page_num + 1
                                    })
                                else:
                                    # Add to current section
                                    current_section['content'] += line_text + " "
                                
                                # Add to full text
                                document_structure['full_text'] += line_text + " "
            
            # Don't forget the last section
            if current_section['content'].strip():
                current_section['page_end'] = len(doc)
                document_structure['sections'].append(current_section)
            
            doc.close()
            return document_structure
            
        except Exception as e:
            logger.error(f"Error extracting structure from {pdf_path.name}: {e}")
            return self._fallback_extraction(pdf_path)
    
    def _is_heading(self, text: str, fonts: List[Dict]) -> Tuple[bool, int]:
        """Determine if text is a heading and its level"""
        if not text or len(text.strip()) < 3:
            return False, 0
        
        # Pattern-based detection
        for pattern in self.heading_patterns:
            if re.match(pattern, text.upper().strip()):
                return True, self._determine_heading_level(text)
        
        # Font-based detection
        if fonts:
            avg_size = sum(f['size'] for f in fonts) / len(fonts)
            is_bold = any(f['flags'] & 2**4 for f in fonts)  # Bold flag
            
            # Heuristic: larger font or bold text that's short enough
            if (avg_size > 12 or is_bold) and len(text) < 100:
                return True, self._determine_heading_level(text)
        
        return False, 0
    
    def _determine_heading_level(self, text: str) -> int:
        """Determine heading level based on content"""
        text_upper = text.upper().strip()
        
        # Level 1: Major sections
        if any(section in text_upper for section in ['INTRODUCTION', 'ABSTRACT', 'CONCLUSION', 'DISCUSSION']):
            return 1
        
        # Level 2: Numbered sections
        if re.match(r'^[0-9]+\.', text_upper):
            return 2
        
        # Level 3: Subsections
        if re.match(r'^[0-9]+\.[0-9]+', text_upper):
            return 3
        
        return 2  # Default
    
    def _fallback_extraction(self, pdf_path: Path) -> Dict:
        """Fallback to simple text extraction if structure detection fails"""
        try:
            with open(pdf_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                full_text = ""
                for page in pdf_reader.pages:
                    full_text += page.extract_text() + " "
                
                return {
                    'filename': pdf_path.name,
                    'sections': [{
                        'heading': 'Full Document',
                        'level': 1,
                        'content': full_text,
                        'page_start': 1,
                        'page_end': len(pdf_reader.pages)
                    }],
                    'full_text': full_text,
                    'headings': [{'text': 'Full Document', 'level': 1, 'page': 1}],
                    'fonts': {}
                }
        except Exception as e:
            logger.error(f"Fallback extraction failed for {pdf_path.name}: {e}")
            return None

class SemanticChunker:
    """Advanced semantic chunking with medical text awareness"""
    
    def __init__(self):
        # Load sentence transformer for embeddings
        try:
            self.embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
        except Exception as e:
            logger.warning(f"Could not load embedding model: {e}")
            self.embedding_model = None
        
        # Load spaCy model for better sentence segmentation
        try:
            self.nlp = spacy.load("en_core_web_sm")
        except OSError:
            logger.warning("spaCy model not found. Using NLTK fallback.")
            self.nlp = None
        
        # Medical discourse markers
        self.section_markers = {
            'case_presentation': ['case presentation', 'patient', 'admitted', 'presented with'],
            'diagnosis': ['diagnosis', 'diagnostic', 'differential diagnosis', 'confirmed by'],
            'treatment': ['treatment', 'therapy', 'medication', 'administered', 'prescribed'],
            'discussion': ['discussion', 'in conclusion', 'this case illustrates'],
            'methodology': ['methods', 'procedure', 'protocol', 'conducted'],
            'results': ['results', 'findings', 'observed', 'demonstrated']
        }
    
    def create_semantic_chunks(self, document_structure: Dict) -> List[Dict]:
        """Create semantically coherent chunks from document structure"""
        all_chunks = []
        
        # Process each section separately to maintain topical purity
        for section_idx, section in enumerate(document_structure['sections']):
            section_chunks = self._chunk_section(
                section, 
                document_structure['filename'],
                section_idx
            )
            all_chunks.extend(section_chunks)
        
        return all_chunks
    
    def _chunk_section(self, section: Dict, filename: str, section_idx: int) -> List[Dict]:
        """Chunk a single section maintaining semantic coherence"""
        content = section['content'].strip()
        if len(content) < MIN_CHUNK_SIZE:
            return []  # Skip very short sections
        
        # First: sentence-level segmentation
        sentences = self._segment_sentences(content)
        if len(sentences) < 2:
            return []
        
        # Method 1: Embedding-based clustering (if model available)
        if self.embedding_model:
            chunks = self._embedding_based_chunking(sentences, section, filename, section_idx)
        else:
            # Method 2: Rule-based chunking fallback
            chunks = self._rule_based_chunking(sentences, section, filename, section_idx)
        
        return chunks
    
    def _segment_sentences(self, text: str) -> List[str]:
        """Segment text into sentences using spaCy or NLTK"""
        if self.nlp:
            doc = self.nlp(text)
            return [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 20]
        else:
            sentences = sent_tokenize(text)
            return [sent.strip() for sent in sentences if len(sent.strip()) > 20]
    
    def _embedding_based_chunking(self, sentences: List[str], section: Dict, 
                                filename: str, section_idx: int) -> List[Dict]:
        """Use embeddings to create semantically coherent chunks"""
        if len(sentences) < 2:
            return []
        
        try:
            # Compute sentence embeddings
            embeddings = self.embedding_model.encode(sentences)
            
            # Agglomerative clustering based on semantic similarity
            # Dynamically determine number of clusters based on content length
            total_chars = sum(len(s) for s in sentences)
            target_clusters = max(1, total_chars // MAX_CHUNK_SIZE)
            target_clusters = min(target_clusters, len(sentences))
            
            if target_clusters == 1:
                clusters = [0] * len(sentences)
            else:
                clustering = AgglomerativeClustering(
                    n_clusters=target_clusters,
                    metric='cosine',
                    linkage='average'
                )
                clusters = clustering.fit_predict(embeddings)
            
            # Group sentences by cluster
            chunk_groups = {}
            for i, cluster_id in enumerate(clusters):
                if cluster_id not in chunk_groups:
                    chunk_groups[cluster_id] = []
                chunk_groups[cluster_id].append((i, sentences[i]))
            
            # Create chunks from groups
            chunks = []
            for cluster_id, sentence_group in chunk_groups.items():
                # Sort by original order
                sentence_group.sort(key=lambda x: x[0])
                
                chunk_text = ' '.join([sent[1] for sent in sentence_group])
                
                # Skip if too small or too large
                if len(chunk_text) < MIN_CHUNK_SIZE:
                    continue
                    
                # Split large chunks further
                if len(chunk_text) > MAX_CHUNK_SIZE:
                    sub_chunks = self._split_large_chunk(chunk_text)
                    for sub_chunk in sub_chunks:
                        chunks.append(self._create_chunk_dict(
                            sub_chunk, section, filename, section_idx, 
                            len(chunks), 'embedding_split'
                        ))
                else:
                    chunks.append(self._create_chunk_dict(
                        chunk_text, section, filename, section_idx,
                        len(chunks), 'embedding_cluster'
                    ))
            
            return chunks
            
        except Exception as e:
            logger.warning(f"Embedding-based chunking failed: {e}. Using fallback.")
            return self._rule_based_chunking(sentences, section, filename, section_idx)
    
    def _rule_based_chunking(self, sentences: List[str], section: Dict,
                           filename: str, section_idx: int) -> List[Dict]:
        """Fallback rule-based chunking with medical discourse awareness"""
        chunks = []
        current_chunk = ""
        current_type = None
        
        for sentence in sentences:
            # Detect discourse type
            sentence_type = self._detect_discourse_type(sentence)
            
            # Start new chunk if discourse type changes or size limit reached
            if (current_type and sentence_type != current_type) or \
               (len(current_chunk) + len(sentence) > MAX_CHUNK_SIZE):
                
                if len(current_chunk.strip()) >= MIN_CHUNK_SIZE:
                    chunks.append(self._create_chunk_dict(
                        current_chunk.strip(), section, filename, section_idx,
                        len(chunks), 'rule_based', current_type
                    ))
                
                current_chunk = sentence + " "
                current_type = sentence_type
            else:
                current_chunk += sentence + " "
                if not current_type:
                    current_type = sentence_type
        
        # Don't forget the last chunk
        if len(current_chunk.strip()) >= MIN_CHUNK_SIZE:
            chunks.append(self._create_chunk_dict(
                current_chunk.strip(), section, filename, section_idx,
                len(chunks), 'rule_based', current_type
            ))
        
        return chunks
    
    def _detect_discourse_type(self, sentence: str) -> str:
        """Detect medical discourse type of sentence"""
        sentence_lower = sentence.lower()
        
        for discourse_type, markers in self.section_markers.items():
            if any(marker in sentence_lower for marker in markers):
                return discourse_type
        
        return 'general_medical'
    
    def _split_large_chunk(self, text: str) -> List[str]:
        """Split overly large chunks while respecting sentence boundaries"""
        sentences = self._segment_sentences(text)
        chunks = []
        current_chunk = ""
        
        for sentence in sentences:
            if len(current_chunk) + len(sentence) > MAX_CHUNK_SIZE and current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = sentence + " "
            else:
                current_chunk += sentence + " "
        
        if current_chunk.strip():
            chunks.append(current_chunk.strip())
        
        return chunks
    
    def _create_chunk_dict(self, content: str, section: Dict, filename: str,
                          section_idx: int, chunk_idx: int, method: str,
                          discourse_type: str = None) -> Dict:
        """Create a standardized chunk dictionary"""
        chunk_id = f"{Path(filename).stem}_s{section_idx}_c{chunk_idx}"
        
        return {
            'chunk_id': chunk_id,
            'source_file': filename,
            'section_heading': section['heading'],
            'section_level': section['level'],
            'content': content,
            'char_count': len(content),
            'word_count': len(content.split()),
            'chunking_method': method,
            'discourse_type': discourse_type or 'unknown',
            'section_pages': f"{section['page_start']}-{section['page_end']}",
            'created_at': datetime.now().isoformat(),
            'hash': hashlib.md5(content.encode()).hexdigest()[:12]
        }

class LLMEnricher:
    """LLM-powered chunk enrichment using Gemini"""
    
    def __init__(self):
        self.model = None
        try:
            # Initialize Gemini model
            self.model = genai.GenerativeModel(
                LLM_MODEL_NAME,
                safety_settings={
                    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
                    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                }
            )
        except Exception as e:
            logger.warning(f"Could not initialize LLM model: {e}")
        
        # Few-shot examples for better prompting
        self.few_shot_examples = [
            {
                "input": "Leishmaniasis is a vector-borne disease caused by protozoan parasites of the genus Leishmania and transmitted by phlebotomine sandflies. The disease is endemic in tropical and subtropical regions worldwide.",
                "output": {
                    "summary": "Leishmaniasis is a sandfly-transmitted parasitic disease endemic to tropical regions.",
                    "medical_keywords": ["leishmaniasis", "vector-borne disease", "Leishmania", "protozoan parasites", "phlebotomine sandflies", "endemic", "tropical", "subtropical"],
                    "content_type": "disease_definition",
                    "leishmania_relevance": 1.0,
                    "clinical_significance": "high"
                }
            },
            {
                "input": "The patient was a 45-year-old male who presented with fever, hepatosplenomegaly, and weight loss. Laboratory findings revealed pancytopenia and elevated liver enzymes.",
                "output": {
                    "summary": "Middle-aged male presented with systemic symptoms and abnormal lab findings.",
                    "medical_keywords": ["fever", "hepatosplenomegaly", "weight loss", "pancytopenia", "liver enzymes", "clinical presentation"],
                    "content_type": "case_presentation",
                    "leishmania_relevance": 0.3,
                    "clinical_significance": "medium"
                }
            }
        ]
    
    def enrich_chunks(self, chunks: List[Dict]) -> List[Dict]:
        """Enrich chunks with LLM-powered metadata"""
        if not self.model:
            logger.warning("LLM model not available. Returning chunks without enrichment.")
            return chunks
        
        enriched_chunks = []
        
        for i, chunk in enumerate(chunks):
            try:
                logger.info(f"Enriching chunk {i+1}/{len(chunks)}: {chunk['chunk_id']}")
                
                enrichment = self._enrich_single_chunk(chunk['content'])
                
                # Add enrichment to chunk
                chunk.update({
                    'llm_summary': enrichment.get('summary', ''),
                    'medical_keywords': enrichment.get('medical_keywords', []),
                    'llm_content_type': enrichment.get('content_type', 'unknown'),
                    'leishmania_relevance': enrichment.get('leishmania_relevance', 0.0),
                    'clinical_significance': enrichment.get('clinical_significance', 'unknown'),
                    'enrichment_timestamp': datetime.now().isoformat()
                })
                
                # Only keep chunks with sufficient relevance
                if enrichment.get('leishmania_relevance', 0) >= MIN_RELEVANCE_SCORE:
                    enriched_chunks.append(chunk)
                else:
                    logger.debug(f"Filtered out chunk {chunk['chunk_id']} (relevance: {enrichment.get('leishmania_relevance', 0)})")
                    
            except Exception as e:
                logger.error(f"Failed to enrich chunk {chunk['chunk_id']}: {e}")
                # Keep chunk without enrichment
                chunk.update({
                    'llm_summary': '',
                    'medical_keywords': [],
                    'llm_content_type': 'unknown',
                    'leishmania_relevance': 0.0,
                    'clinical_significance': 'unknown',
                    'enrichment_error': str(e)
                })
                enriched_chunks.append(chunk)
                
            # --- RATE LIMIT HERE (sleep for 4.2s per call for 15 calls/min max) ---
            if i < len(chunks) - 1:
                time.sleep(4.2)

        
        logger.info(f"Enrichment complete. Kept {len(enriched_chunks)}/{len(chunks)} chunks.")
        return enriched_chunks
    
    def _enrich_single_chunk(self, content: str) -> Dict:
        """Enrich a single chunk using LLM"""
        prompt = self._create_enrichment_prompt(content)
        
        try:
            response = self.model.generate_content(prompt)
            
            # Parse JSON response
            response_text = response.text.strip()
            if response_text.startswith('```json'):
                response_text = response_text[7:]
            if response_text.endswith('```'):
                response_text = response_text[:-3]
            
            enrichment = json.loads(response_text.strip())
            
            # Validate and clean the response
            return self._validate_enrichment(enrichment)
            
        except Exception as e:
            logger.error(f"LLM enrichment failed: {e}")
            return {
                'summary': content[:100] + "...",
                'medical_keywords': [],
                'content_type': 'unknown',
                'leishmania_relevance': 0.0,
                'clinical_significance': 'unknown'
            }
    
    def _create_enrichment_prompt(self, content: str) -> str:
        """Create enrichment prompt with few-shot examples"""
        
        examples_text = ""
        for example in self.few_shot_examples[:2]:  # Use first 2 examples
            examples_text += f"""
Input: "{example['input']}"
Output: {json.dumps(example['output'], indent=2)}

"""
        
        prompt = f"""You are a medical AI assistant specializing in analyzing medical texts for information about Leishmaniasis and related parasitic diseases.

Your task is to analyze the given medical text chunk and provide structured metadata in JSON format.

{examples_text}

Now analyze this text chunk:
Input: "{content}"

Provide your analysis in the following JSON format:
{{
    "summary": "One-sentence summary of the main point",
    "medical_keywords": ["list", "of", "relevant", "medical", "terms"],
    "content_type": "one of: disease_definition, pathophysiology, clinical_presentation, diagnosis, treatment, epidemiology, case_report, general_medical, reference",
    "leishmania_relevance": 0.0-1.0 (how relevant to Leishmaniasis specifically),
    "clinical_significance": "high/medium/low"
}}

Rules:
- Be precise and concise
- Extract only genuinely relevant medical keywords
- Rate leishmania_relevance strictly: 1.0 = directly about Leishmania, 0.8 = highly related, 0.5 = somewhat related, 0.0 = unrelated
- Return only valid JSON, no explanations

Output:"""

        return prompt
    
    def _validate_enrichment(self, enrichment: Dict) -> Dict:
        """Validate and clean enrichment response"""
        validated = {
            'summary': str(enrichment.get('summary', ''))[:200],  # Limit length
            'medical_keywords': [],
            'content_type': 'unknown',
            'leishmania_relevance': 0.0,
            'clinical_significance': 'unknown'
        }
        
        # Validate keywords
        if 'medical_keywords' in enrichment and isinstance(enrichment['medical_keywords'], list):
            validated['medical_keywords'] = [
                str(kw).lower().strip() 
                for kw in enrichment['medical_keywords'][:20]  # Limit to 20 keywords
                if len(str(kw).strip()) > 2
            ]
        
        # Validate content type
        valid_content_types = [
            'disease_definition', 'pathophysiology', 'clinical_presentation',
            'diagnosis', 'treatment', 'epidemiology', 'case_report', 
            'general_medical', 'reference'
        ]
        if enrichment.get('content_type') in valid_content_types:
            validated['content_type'] = enrichment['content_type']
        
        # Validate relevance score
        try:
            relevance = float(enrichment.get('leishmania_relevance', 0))
            validated['leishmania_relevance'] = max(0.0, min(1.0, relevance))
        except (ValueError, TypeError):
            validated['leishmania_relevance'] = 0.0
        
        # Validate clinical significance
        valid_significance = ['high', 'medium', 'low']
        if enrichment.get('clinical_significance') in valid_significance:
            validated['clinical_significance'] = enrichment['clinical_significance']
        
        return validated

class EnhancedTextbookProcessor:
    """Main enhanced processor orchestrating all components"""
    
    def __init__(self):
        self.setup_directories()
        self.heading_extractor = HeadingExtractor()
        self.semantic_chunker = SemanticChunker()
        self.llm_enricher = LLMEnricher()
        self.processed_files = []
        
    def setup_directories(self):
        """Setup enhanced directory structure"""
        directories = [
            RAG_OUTPUT_DIR / "chunks",
            RAG_OUTPUT_DIR / "images",
            RAG_OUTPUT_DIR / "metadata",
            FINETUNE_OUTPUT_DIR / "qa_pairs",
            FINETUNE_OUTPUT_DIR / "summaries",
            FINETUNE_OUTPUT_DIR / "multimodal_pairs",
            SEMANTIC_CHUNKS_DIR,
            ENRICHED_CHUNKS_DIR,
            ENRICHED_CHUNKS_DIR / "core",  # High relevance chunks
            ENRICHED_CHUNKS_DIR / "longtail"  # Medium relevance chunks
        ]
        
        for dir_path in directories:
            dir_path.mkdir(parents=True, exist_ok=True)
    
    def process_single_file(self, pdf_path: Path) -> Dict:
        """Process a single PDF with enhanced pipeline"""
        start_time = time.time()
        
        try:
            logger.info(f"🚀 Starting enhanced processing: {pdf_path.name}")
            
            # Step 1: Extract document structure with headings
            logger.info(f"📖 Extracting document structure...")
            document_structure = self.heading_extractor.extract_document_structure(pdf_path)
            if not document_structure:
                return {
                    'filename': pdf_path.name,
                    'status': 'failed',
                    'error': 'Failed to extract document structure'
                }
            
            # Step 2: Create semantic chunks
            logger.info(f"🧩 Creating semantic chunks...")
            raw_chunks = self.semantic_chunker.create_semantic_chunks(document_structure)
            
            if not raw_chunks:
                return {
                    'filename': pdf_path.name,
                    'status': 'failed',
                    'error': 'No valid chunks created'
                }
            
            # Save raw semantic chunks
            self._save_semantic_chunks(raw_chunks, pdf_path.stem)
            
            # Step 3: LLM enrichment
            logger.info(f"✨ Enriching chunks with LLM...")
            enriched_chunks = self.llm_enricher.enrich_chunks(raw_chunks)
            
            # Step 4: Tier chunks by relevance
            core_chunks = [c for c in enriched_chunks if c.get('leishmania_relevance', 0) >= HIGH_RELEVANCE_THRESHOLD]
            longtail_chunks = [c for c in enriched_chunks if MIN_RELEVANCE_SCORE <= c.get('leishmania_relevance', 0) < HIGH_RELEVANCE_THRESHOLD]
            
            # Save enriched chunks
            self._save_enriched_chunks(core_chunks, longtail_chunks, pdf_path.stem)
            
            # Step 5: Generate enhanced Q&A pairs from core chunks
            qa_pairs = self._generate_enhanced_qa_pairs(core_chunks, document_structure)
            self._save_qa_data(qa_pairs, pdf_path.stem)
            
            # Step 6: Create document summary
            summary_data = self._create_enhanced_summary(document_structure, core_chunks, longtail_chunks)
            self._save_summary_data(summary_data, pdf_path.stem)
            
            processing_time = time.time() - start_time
            
            result = {
                'filename': pdf_path.name,
                'filepath': str(pdf_path),
                'status': 'success',
                'processed_at': datetime.now().isoformat(),
                'processing_time': processing_time,
                'document_structure': {
                    'total_sections': len(document_structure['sections']),
                    'total_headings': len(document_structure['headings']),
                    'total_characters': len(document_structure['full_text'])
                },
                'chunking_stats': {
                    'raw_chunks_created': len(raw_chunks),
                    'chunks_after_enrichment': len(enriched_chunks),
                    'core_chunks': len(core_chunks),
                    'longtail_chunks': len(longtail_chunks),
                    'filtered_out': len(raw_chunks) - len(enriched_chunks)
                },
                'qa_pairs_created': len(qa_pairs),
                'avg_chunk_relevance': sum(c.get('leishmania_relevance', 0) for c in enriched_chunks) / len(enriched_chunks) if enriched_chunks else 0,
                'high_relevance_ratio': len(core_chunks) / len(enriched_chunks) if enriched_chunks else 0
            }
            
            logger.info(f"✅ Enhanced processing complete: {pdf_path.name} "
                       f"({processing_time:.1f}s, {len(core_chunks)} core chunks)")
            
            return result
            
        except Exception as e:
            error_msg = f"Enhanced processing failed for {pdf_path.name}: {str(e)}"
            logger.error(error_msg)
            
            return {
                'filename': pdf_path.name,
                'filepath': str(pdf_path),
                'status': 'failed',
                'error': str(e),
                'processed_at': datetime.now().isoformat(),
                'processing_time': time.time() - start_time
            }
    
    def _save_semantic_chunks(self, chunks: List[Dict], filename_base: str):
        """Save semantic chunks before enrichment"""
        output_file = SEMANTIC_CHUNKS_DIR / f"{filename_base}_semantic_chunks.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(chunks, f, ensure_ascii=False, indent=2)
    
    def _save_enriched_chunks(self, core_chunks: List[Dict], longtail_chunks: List[Dict], filename_base: str):
        """Save enriched chunks in tiered structure"""
        # Save core chunks (high relevance)
        core_file = ENRICHED_CHUNKS_DIR / "core" / f"{filename_base}_core_chunks.json"
        with open(core_file, 'w', encoding='utf-8') as f:
            json.dump(core_chunks, f, ensure_ascii=False, indent=2)
        
        # Save longtail chunks (medium relevance)
        if longtail_chunks:
            longtail_file = ENRICHED_CHUNKS_DIR / "longtail" / f"{filename_base}_longtail_chunks.json"
            with open(longtail_file, 'w', encoding='utf-8') as f:
                json.dump(longtail_chunks, f, ensure_ascii=False, indent=2)
        
        # Save combined chunks for legacy compatibility
        all_chunks = core_chunks + longtail_chunks
        combined_file = RAG_OUTPUT_DIR / "chunks" / f"{filename_base}_chunks.json"
        with open(combined_file, 'w', encoding='utf-8') as f:
            json.dump(all_chunks, f, ensure_ascii=False, indent=2)
    
    def _generate_enhanced_qa_pairs(self, chunks: List[Dict], document_structure: Dict) -> List[Dict]:
        """Generate higher quality Q&A pairs from enriched chunks"""
        qa_pairs = []
        
        # Group chunks by content type for better question generation
        chunks_by_type = {}
        for chunk in chunks:
            content_type = chunk.get('llm_content_type', 'unknown')
            if content_type not in chunks_by_type:
                chunks_by_type[content_type] = []
            chunks_by_type[content_type].append(chunk)
        
        # Generate questions based on content type
        for content_type, type_chunks in chunks_by_type.items():
            for chunk in type_chunks:
                chunk_qa = self._generate_qa_for_chunk(chunk, content_type)
                qa_pairs.extend(chunk_qa)
        
        # Add cross-chunk questions for comprehensive understanding
        if len(chunks) > 1:
            cross_chunk_qa = self._generate_cross_chunk_questions(chunks, document_structure)
            qa_pairs.extend(cross_chunk_qa)
        
        return qa_pairs
    
    def _generate_qa_for_chunk(self, chunk: Dict, content_type: str) -> List[Dict]:
        """Generate Q&A pairs for a single chunk based on its type"""
        qa_pairs = []
        content = chunk['content']
        keywords = chunk.get('medical_keywords', [])
        
        # Type-specific question templates
        question_templates = {
            'disease_definition': [
                f"What is {', '.join(keywords[:2])}?" if keywords else "What disease is being described?",
                "How is this condition defined?",
                "What are the key characteristics mentioned?"
            ],
            'pathophysiology': [
                "What is the underlying mechanism described?",
                "How does this biological process work?",
                f"What role do {', '.join(keywords[:2])} play?" if keywords else "What biological factors are involved?"
            ],
            'clinical_presentation': [
                "What are the clinical signs and symptoms?",
                "How does the patient typically present?",
                "What clinical features are described?"
            ],
            'diagnosis': [
                "What diagnostic methods are mentioned?",
                "How is this condition diagnosed?",
                "What tests are recommended?"
            ],
            'treatment': [
                "What treatment options are described?",
                "What is the recommended therapy?",
                f"How is {', '.join(keywords[:2])} treated?" if keywords else "What medications are mentioned?"
            ],
            'epidemiology': [
                "What is the epidemiological pattern described?",
                "What populations are affected?",
                "What is the geographic distribution?"
            ],
            'case_report': [
                "What was the patient's presentation?",
                "What was the diagnosis and treatment?",
                "What can we learn from this case?"
            ]
        }
        
        templates = question_templates.get(content_type, [
            "What is the main point discussed?",
            "What medical information is provided?",
            "What key concepts are explained?"
        ])
        
        for i, question in enumerate(templates[:3]):  # Limit to 3 questions per chunk
            answer = self._extract_targeted_answer(question, content, keywords)
            
            qa_pair = {
                'question_id': f"{chunk['chunk_id']}_qa_{i}",
                'source_file': chunk['source_file'],
                'chunk_id': chunk['chunk_id'],
                'question': question,
                'answer': answer,
                'context': content,
                'content_type': content_type,
                'question_type': self._classify_question_type(question),
                'relevance_score': chunk.get('leishmania_relevance', 0),
                'keywords': keywords,
                'created_at': datetime.now().isoformat(),
                'generation_method': 'enhanced_template'
            }
            qa_pairs.append(qa_pair)
        
        return qa_pairs
    
    def _generate_cross_chunk_questions(self, chunks: List[Dict], document_structure: Dict) -> List[Dict]:
        """Generate questions that span multiple chunks for comprehensive understanding"""
        cross_qa = []
        
        # Find chunks about different aspects that can be connected
        diagnostic_chunks = [c for c in chunks if c.get('llm_content_type') == 'diagnosis']
        treatment_chunks = [c for c in chunks if c.get('llm_content_type') == 'treatment']
        clinical_chunks = [c for c in chunks if c.get('llm_content_type') == 'clinical_presentation']
        
        # Generate connecting questions
        if diagnostic_chunks and treatment_chunks:
            combined_context = diagnostic_chunks[0]['content'] + " " + treatment_chunks[0]['content']
            cross_qa.append({
                'question_id': f"{document_structure['filename']}_cross_diag_treat",
                'source_file': document_structure['filename'],
                'chunk_id': f"{diagnostic_chunks[0]['chunk_id']},{treatment_chunks[0]['chunk_id']}",
                'question': "What is the relationship between diagnosis and treatment described?",
                'answer': self._extract_targeted_answer("How are diagnosis and treatment connected?", combined_context, []),
                'context': combined_context[:1000] + "..." if len(combined_context) > 1000 else combined_context,
                'content_type': 'cross_reference',
                'question_type': 'relationship',
                'generation_method': 'cross_chunk',
                'created_at': datetime.now().isoformat()
            })
        
        if clinical_chunks and diagnostic_chunks:
            combined_context = clinical_chunks[0]['content'] + " " + diagnostic_chunks[0]['content']
            cross_qa.append({
                'question_id': f"{document_structure['filename']}_cross_clin_diag",
                'source_file': document_structure['filename'],
                'chunk_id': f"{clinical_chunks[0]['chunk_id']},{diagnostic_chunks[0]['chunk_id']}",
                'question': "How do clinical presentations lead to diagnostic considerations?",
                'answer': self._extract_targeted_answer("What clinical features guide diagnosis?", combined_context, []),
                'context': combined_context[:1000] + "..." if len(combined_context) > 1000 else combined_context,
                'content_type': 'cross_reference',
                'question_type': 'causal',
                'generation_method': 'cross_chunk',
                'created_at': datetime.now().isoformat()
            })
        
        return cross_qa
    
    def _extract_targeted_answer(self, question: str, content: str, keywords: List[str]) -> str:
        """Extract targeted answer based on question and keywords"""
        sentences = sent_tokenize(content)
        question_lower = question.lower()
        
        # Extract key question words
        question_words = set(word_tokenize(question_lower)) - set(stopwords.words('english'))
        
        # Score sentences based on relevance
        scored_sentences = []
        for sentence in sentences:
            sentence_lower = sentence.lower()
            sentence_words = set(word_tokenize(sentence_lower))
            
            # Base score from question word overlap
            overlap_score = len(question_words & sentence_words) / len(question_words) if question_words else 0
            
            # Bonus for medical keywords
            keyword_score = sum(1 for kw in keywords if kw in sentence_lower) * 0.3
            
            # Bonus for medical indicators
            medical_indicators = ['treatment', 'diagnosis', 'patient', 'clinical', 'therapy', 'disease', 'condition']
            medical_score = sum(0.2 for indicator in medical_indicators if indicator in sentence_lower)
            
            total_score = overlap_score + keyword_score + medical_score
            
            if total_score > 0 and len(sentence.strip()) > 20:
                scored_sentences.append((sentence.strip(), total_score))
        
        # Sort by score and take top sentences
        scored_sentences.sort(key=lambda x: x[1], reverse=True)
        top_sentences = [sent[0] for sent in scored_sentences[:3]]
        
        if top_sentences:
            answer = '. '.join(top_sentences)
            return answer if len(answer) <= 500 else answer[:500] + "..."
        else:
            # Fallback to first few sentences
            fallback = '. '.join(sentences[:2])
            return fallback if len(fallback) <= 300 else fallback[:300] + "..."
    
    def _classify_question_type(self, question: str) -> str:
        """Classify question type for better organization"""
        question_lower = question.lower().strip()
        
        if question_lower.startswith('what'):
            if any(word in question_lower for word in ['is', 'are', 'definition']):
                return 'definition'
            elif any(word in question_lower for word in ['treatment', 'therapy', 'medication']):
                return 'treatment'
            elif any(word in question_lower for word in ['symptom', 'sign', 'present']):
                return 'clinical'
            else:
                return 'factual'
        elif question_lower.startswith('how'):
            if any(word in question_lower for word in ['diagnos', 'test']):
                return 'diagnostic_procedure'
            elif any(word in question_lower for word in ['treat', 'manage']):
                return 'treatment_procedure'
            else:
                return 'process'
        elif question_lower.startswith('when'):
            return 'temporal'
        elif question_lower.startswith('where'):
            return 'location'
        elif question_lower.startswith('why'):
            return 'causal'
        else:
            return 'general'
    
    def _create_enhanced_summary(self, document_structure: Dict, core_chunks: List[Dict], longtail_chunks: List[Dict]) -> Dict:
        """Create enhanced document summary with multiple perspectives"""
        all_chunks = core_chunks + longtail_chunks
        
        # Aggregate keywords from all chunks
        all_keywords = []
        for chunk in all_chunks:
            all_keywords.extend(chunk.get('medical_keywords', []))
        
        # Count keyword frequencies
        keyword_freq = {}
        for keyword in all_keywords:
            keyword_freq[keyword] = keyword_freq.get(keyword, 0) + 1
        
        # Top keywords
        top_keywords = sorted(keyword_freq.items(), key=lambda x: x[1], reverse=True)[:20]
        
        # Content type distribution
        content_types = {}
        for chunk in all_chunks:
            content_type = chunk.get('llm_content_type', 'unknown')
            content_types[content_type] = content_types.get(content_type, 0) + 1
        
        # Relevance statistics
        relevance_scores = [chunk.get('leishmania_relevance', 0) for chunk in all_chunks]
        avg_relevance = sum(relevance_scores) / len(relevance_scores) if relevance_scores else 0
        
        summary_data = {
            'document_id': document_structure['filename'].replace('.pdf', ''),
            'source_file': document_structure['filename'],
            'processing_method': 'enhanced_semantic',
            'created_at': datetime.now().isoformat(),
            
            # Document structure summary
            'document_structure': {
                'total_sections': len(document_structure['sections']),
                'section_headings': [s['heading'] for s in document_structure['sections']],
                'total_text_length': len(document_structure['full_text']),
                'heading_hierarchy': document_structure['headings']
            },
            
            # Chunk statistics
            'chunk_statistics': {
                'total_chunks': len(all_chunks),
                'core_chunks': len(core_chunks),
                'longtail_chunks': len(longtail_chunks),
                'avg_relevance_score': avg_relevance,
                'content_type_distribution': content_types,
                'avg_chunk_length': sum(c['char_count'] for c in all_chunks) / len(all_chunks) if all_chunks else 0
            },
            
            # Content analysis
            'content_analysis': {
                'top_keywords': [kw[0] for kw in top_keywords[:10]],
                'keyword_frequencies': dict(top_keywords),
                'leishmania_relevance_distribution': {
                    'high (0.8+)': len([c for c in all_chunks if c.get('leishmania_relevance', 0) >= 0.8]),
                    'medium (0.5-0.8)': len([c for c in all_chunks if 0.5 <= c.get('leishmania_relevance', 0) < 0.8]),
                    'low (0.3-0.5)': len([c for c in all_chunks if 0.3 <= c.get('leishmania_relevance', 0) < 0.5])
                }
            },
            
            # Generated summaries at different levels
            'summaries': {
                'executive': self._create_executive_summary(core_chunks),
                'detailed': self._create_detailed_summary(all_chunks, document_structure),
                'core_insights': self._extract_core_insights(core_chunks)
            }
        }
        
        return summary_data
    
    def _create_executive_summary(self, core_chunks: List[Dict]) -> str:
        """Create executive summary from core chunks"""
        if not core_chunks:
            return "No high-relevance content found for executive summary."
        
        # Get the most relevant summaries
        chunk_summaries = []
        for chunk in core_chunks[:5]:  # Top 5 chunks
            summary = chunk.get('llm_summary', '')
            if summary and len(summary) > 20:
                chunk_summaries.append(summary)
        
        if chunk_summaries:
            return ' '.join(chunk_summaries)
        else:
            # Fallback: extract key sentences
            all_content = ' '.join([c['content'] for c in core_chunks[:3]])
            sentences = sent_tokenize(all_content)
            key_sentences = []
            
            for sentence in sentences[:10]:
                if any(term in sentence.lower() for term in ['leishmania', 'treatment', 'diagnosis', 'patient']):
                    key_sentences.append(sentence.strip())
                    if len(key_sentences) >= 3:
                        break
            
            return '. '.join(key_sentences) + '.' if key_sentences else "Executive summary not available."
    
    def _create_detailed_summary(self, all_chunks: List[Dict], document_structure: Dict) -> str:
        """Create detailed summary organized by sections"""
        summary_parts = []
        
        # Group chunks by section
        chunks_by_section = {}
        for chunk in all_chunks:
            section = chunk.get('section_heading', 'Unknown Section')
            if section not in chunks_by_section:
                chunks_by_section[section] = []
            chunks_by_section[section].append(chunk)
        
        # Summarize each section
        for section_name, section_chunks in chunks_by_section.items():
            if len(section_chunks) == 0:
                continue
                
            section_summary = f"**{section_name}**: "
            
            # Get best summaries from this section
            section_summaries = []
            for chunk in section_chunks[:3]:  # Top 3 chunks per section
                if chunk.get('llm_summary'):
                    section_summaries.append(chunk['llm_summary'])
                elif len(chunk['content']) > 100:
                    # Fallback: first sentence
                    first_sentence = sent_tokenize(chunk['content'])[0] if sent_tokenize(chunk['content']) else chunk['content'][:200] + "..."
                    section_summaries.append(first_sentence)
            
            if section_summaries:
                section_summary += ' '.join(section_summaries)
                summary_parts.append(section_summary)
        
        return '\n\n'.join(summary_parts) if summary_parts else "Detailed summary not available."
    
    def _extract_core_insights(self, core_chunks: List[Dict]) -> List[str]:
        """Extract core insights from high-relevance chunks"""
        insights = []
        
        # Group by content type for structured insights
        insights_by_type = {}
        for chunk in core_chunks:
            content_type = chunk.get('llm_content_type', 'unknown')
            if content_type not in insights_by_type:
                insights_by_type[content_type] = []
            
            if chunk.get('llm_summary'):
                insights_by_type[content_type].append(chunk['llm_summary'])
        
        # Create structured insights
        type_labels = {
            'disease_definition': 'Disease Definition',
            'pathophysiology': 'Pathophysiology',
            'clinical_presentation': 'Clinical Presentation',
            'diagnosis': 'Diagnostic Approach',
            'treatment': 'Treatment Options',
            'epidemiology': 'Epidemiological Context',
            'case_report': 'Case Study Findings'
        }
        
        for content_type, summaries in insights_by_type.items():
            if summaries:
                label = type_labels.get(content_type, content_type.title().replace('_', ' '))
                insight = f"{label}: {' '.join(summaries[:2])}"  # Top 2 summaries per type
                insights.append(insight)
        
        return insights[:8]  # Limit to 8 key insights
    
    def _save_qa_data(self, qa_pairs: List[Dict], filename: str):
        """Save enhanced Q&A pairs"""
        output_dir = FINETUNE_OUTPUT_DIR / "qa_pairs"
        output_dir.mkdir(parents=True, exist_ok=True)
        
        output_file = output_dir / f"{filename}_enhanced_qa.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(qa_pairs, f, ensure_ascii=False, indent=2)
        
        # Also save in training format
        training_format = []
        for qa in qa_pairs:
            training_format.append({
                "instruction": qa['question'],
                "input": qa.get('context', '')[:500],  # Limit context
                "output": qa['answer'],
                "metadata": {
                    "source": qa['source_file'],
                    "content_type": qa.get('content_type', 'unknown'),
                    "relevance_score": qa.get('relevance_score', 0),
                    "question_type": qa.get('question_type', 'general')
                }
            })
        
        training_file = output_dir / f"{filename}_training_format.json"
        with open(training_file, 'w', encoding='utf-8') as f:
            json.dump(training_format, f, ensure_ascii=False, indent=2)
    
    def _save_summary_data(self, summary_data: Dict, filename: str):
        """Save enhanced summary data"""
        output_dir = FINETUNE_OUTPUT_DIR / "summaries"
        output_dir.mkdir(parents=True, exist_ok=True)
        
        output_file = output_dir / f"{filename}_enhanced_summary.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(summary_data, f, ensure_ascii=False, indent=2)
    
    def process_all_files(self):
        """Process all files with enhanced pipeline"""
        if not TEXTBOOK_SOURCE_DIR.exists():
            logger.error(f"❌ Source directory not found: {TEXTBOOK_SOURCE_DIR}")
            return
        
        pdf_files = list(TEXTBOOK_SOURCE_DIR.rglob('*.pdf'))
        total_files = len(pdf_files)
        
        if total_files == 0:
            logger.warning("⚠️ No PDF files found!")
            return
        
        logger.info(f"🚀 ENHANCED PROCESSING PIPELINE")
        logger.info(f"📚 Found {total_files} PDF files")
        logger.info(f"🧩 Semantic chunking enabled: {self.semantic_chunker.embedding_model is not None}")
        logger.info(f"✨ LLM enrichment enabled: {self.llm_enricher.model is not None}")
        logger.info(f"📊 Relevance threshold: {MIN_RELEVANCE_SCORE}")
        logger.info(f"🎯 High relevance threshold: {HIGH_RELEVANCE_THRESHOLD}")
        
        start_time = time.time()
        successful_files = 0
        failed_files = 0
        
        total_stats = {
            'total_raw_chunks': 0,
            'total_enriched_chunks': 0,
            'total_core_chunks': 0,
            'total_longtail_chunks': 0,
            'total_qa_pairs': 0,
            'avg_relevance_scores': []
        }
        
        for i, pdf_file in enumerate(pdf_files):
            logger.info(f"\n📖 Processing {i+1}/{total_files}: {pdf_file.name}")
            
            result = self.process_single_file(pdf_file)
            self.processed_files.append(result)
            
            if result['status'] == 'success':
                successful_files += 1
                
                # Aggregate statistics
                chunking_stats = result.get('chunking_stats', {})
                total_stats['total_raw_chunks'] += chunking_stats.get('raw_chunks_created', 0)
                total_stats['total_enriched_chunks'] += chunking_stats.get('chunks_after_enrichment', 0)
                total_stats['total_core_chunks'] += chunking_stats.get('core_chunks', 0)
                total_stats['total_longtail_chunks'] += chunking_stats.get('longtail_chunks', 0)
                total_stats['total_qa_pairs'] += result.get('qa_pairs_created', 0)
                
                if 'avg_chunk_relevance' in result:
                    total_stats['avg_relevance_scores'].append(result['avg_chunk_relevance'])
                
            else:
                failed_files += 1
                logger.error(f"❌ Failed: {result.get('error', 'Unknown error')}")
        
        # Calculate final statistics
        total_time = time.time() - start_time
        overall_avg_relevance = (sum(total_stats['avg_relevance_scores']) / 
                               len(total_stats['avg_relevance_scores'])) if total_stats['avg_relevance_scores'] else 0
        
        # Save processing metadata
        self._save_processing_metadata()
        
        # Print final results
        self._print_enhanced_results(total_time, successful_files, failed_files, total_stats, overall_avg_relevance)
    
    def _save_processing_metadata(self):
        """Save enhanced processing metadata"""
        try:
            df_metadata = pd.DataFrame(self.processed_files)
            df_metadata.to_csv(PROCESSED_METADATA_PATH, index=False)
            logger.info(f"💾 Saved processing metadata: {len(self.processed_files)} records")
        except Exception as e:
            logger.error(f"❌ Error saving metadata: {e}")
    
    def _print_enhanced_results(self, total_time: float, successful: int, failed: int, 
                              stats: Dict, avg_relevance: float):
        """Print comprehensive results"""
        print(f"\n{'='*80}")
        print(f"🎉 ENHANCED PROCESSING COMPLETE!")
        print(f"{'='*80}")
        
        print(f"⏱️ Total processing time: {total_time/60:.1f} minutes")
        print(f"✅ Successful: {successful} files")
        print(f"❌ Failed: {failed} files")
        print(f"📈 Success rate: {(successful/(successful+failed)*100):.1f}%")
        
        print(f"\n📊 CHUNKING STATISTICS:")
        print(f"   🧩 Raw chunks created: {stats['total_raw_chunks']:,}")
        print(f"   ✨ Chunks after enrichment: {stats['total_enriched_chunks']:,}")
        print(f"   🎯 Core chunks (high relevance): {stats['total_core_chunks']:,}")
        print(f"   📋 Longtail chunks (medium relevance): {stats['total_longtail_chunks']:,}")
        print(f"   🗑️ Filtered out (low relevance): {stats['total_raw_chunks'] - stats['total_enriched_chunks']:,}")
        print(f"   ❓ Q&A pairs generated: {stats['total_qa_pairs']:,}")
        
        print(f"\n🎯 QUALITY METRICS:")
        print(f"   📊 Average relevance score: {avg_relevance:.3f}")
        retention_rate = (stats['total_enriched_chunks'] / stats['total_raw_chunks'] * 100) if stats['total_raw_chunks'] > 0 else 0
        print(f"   💎 Chunk retention rate: {retention_rate:.1f}%")
        core_ratio = (stats['total_core_chunks'] / stats['total_enriched_chunks'] * 100) if stats['total_enriched_chunks'] > 0 else 0
        print(f"   🏆 Core chunk ratio: {core_ratio:.1f}%")
        
        print(f"\n📁 OUTPUT STRUCTURE:")
        print(f"   🧩 Semantic chunks: {SEMANTIC_CHUNKS_DIR}")
        print(f"   🎯 Core chunks: {ENRICHED_CHUNKS_DIR / 'core'}")
        print(f"   📋 Longtail chunks: {ENRICHED_CHUNKS_DIR / 'longtail'}")
        print(f"   ❓ Enhanced Q&A: {FINETUNE_OUTPUT_DIR / 'qa_pairs'}")
        print(f"   📝 Enhanced summaries: {FINETUNE_OUTPUT_DIR / 'summaries'}")
        print(f"   📊 Processing metadata: {PROCESSED_METADATA_PATH}")
        
        if failed > 0:
            print(f"\n❌ FAILED FILES:")
            for result in self.processed_files:
                if result['status'] == 'failed':
                    print(f"   - {result['filename']}: {result.get('error', 'Unknown error')}")

# --- Utility Functions ---
def quick_enhanced_status():
    """Quick status check for enhanced processing, handling potentially complex CSV columns."""
    if not PROCESSED_METADATA_PATH.exists():
        print("📊 No processing metadata found. Looks like a fresh run.")
        return None

    try:
        df = pd.read_csv(PROCESSED_METADATA_PATH)
        successful_df = df[df['status'] == 'success'].copy()
        
        total_files = len(df)
        successful_count = len(successful_df)
        
        print(f"📊 Enhanced Processing Status:")
        print(f"   ✅ Successful: {successful_count}/{total_files} files")
        
        if successful_count > 0:
            # Safely parse string representations of dictionaries/numbers from CSV
            def safe_extract_sum(column_name: str) -> int:
                if column_name in successful_df.columns:
                    # Fill NaNs with 0 and convert to integer for summation
                    return int(pd.to_numeric(successful_df[column_name], errors='coerce').fillna(0).sum())
                return 0

            def safe_extract_avg(column_name: str) -> float:
                if column_name in successful_df.columns:
                    # Convert to numeric, drop errors/NaNs, then calculate mean
                    numeric_series = pd.to_numeric(successful_df[column_name], errors='coerce').dropna()
                    return numeric_series.mean() if not numeric_series.empty else 0.0
                return 0.0
            
            # Use the safe extraction functions
            total_core = safe_extract_sum('chunking_stats.core_chunks')
            total_longtail = safe_extract_sum('chunking_stats.longtail_chunks')
            total_qa = safe_extract_sum('qa_pairs_created')
            avg_relevance = safe_extract_avg('avg_chunk_relevance')

            print("\n   --- AGGREGATE RESULTS ---")
            print(f"   🎯 Total Core Chunks: {total_core:,}")
            print(f"   📋 Total Longtail Chunks: {total_longtail:,}")
            print(f"   ❓ Total Q&A Pairs Generated: {total_qa:,}")
            print(f"   📊 Overall Average Relevance Score: {avg_relevance:.3f}")

        # Check for failed files
        failed_df = df[df['status'] == 'failed']
        if not failed_df.empty:
            print(f"\n   ❌ Failed Files ({len(failed_df)}):")
            for _, row in failed_df.iterrows():
                error_preview = str(row.get('error', 'Unknown error'))[:70]
                print(f"      - {row['filename']}: {error_preview}...")

        return df

    except Exception as e:
        logger.error(f"❌ Error reading status from {PROCESSED_METADATA_PATH}: {e}")
        return None

def reset_processing():
    """
    Resets the entire enhanced processing pipeline by deleting all generated files and directories.
    """
    print("⚠️ WARNING: This will permanently delete all processed data!")
    print("This includes semantic chunks, enriched data, summaries, Q&A pairs, and metadata.")
    
    confirm = input("Type 'reset' to confirm: ")
    if confirm.lower() != 'reset':
        print("🚫 Reset cancelled.")
        return

    try:
        # Directories to remove
        directories_to_remove = [
            RAG_OUTPUT_DIR, 
            FINETUNE_OUTPUT_DIR,
            SEMANTIC_CHUNKS_DIR,
            ENRICHED_CHUNKS_DIR
        ]
        
        for dir_path in directories_to_remove:
            if dir_path.exists():
                shutil.rmtree(dir_path)
                print(f"🗑️ Deleted directory: {dir_path}")
        
        # Individual files to remove
        files_to_remove = [
            PROCESSED_METADATA_PATH, 
            CHECKPOINT_PATH, 
            FILE_HASHES_PATH
        ]
        
        for file_path in files_to_remove:
            if file_path.exists():
                file_path.unlink()
                print(f"🗑️ Deleted file: {file_path}")
                
        print("\n✅ Reset complete! The environment is clean for a fresh run.")
        
    except Exception as e:
        print(f"❌ An error occurred during reset: {e}")

def show_enhanced_resume_info():
    """Displays a detailed summary of the current state before starting a run."""
    print("\n" + "="*60)
    print("🚀 ENHANCED PIPELINE - RESUME & CONFIGURATION INFO")
    print("="*60)
    
    # Show status from previous run
    quick_enhanced_status()

    # Show configuration
    print("\n   --- CURRENT CONFIGURATION ---")
    print(f"   Embedding Model: {EMBEDDING_MODEL_NAME}")
    print(f"   LLM Model: {LLM_MODEL_NAME}")
    print(f"   Relevance Thresholds: Min={MIN_RELEVANCE_SCORE}, Core={HIGH_RELEVANCE_THRESHOLD}")
    print(f"   Chunk Size (Chars): Min={MIN_CHUNK_SIZE}, Max={MAX_CHUNK_SIZE}")
    print(f"   Max Workers: {MAX_WORKERS}")

    # Check for output directories
    print("\n   --- OUTPUT DIRECTORIES ---")
    core_dir = ENRICHED_CHUNKS_DIR / 'core'
    longtail_dir = ENRICHED_CHUNKS_DIR / 'longtail'
    print(f"   Core Chunks: {'Exists' if core_dir.exists() else 'Not Found'} -> {core_dir}")
    print(f"   Longtail Chunks: {'Exists' if longtail_dir.exists() else 'Not Found'} -> {longtail_dir}")
    print(f"   Q&A Pairs: {'Exists' if (FINETUNE_OUTPUT_DIR / 'qa_pairs').exists() else 'Not Found'} -> {FINETUNE_OUTPUT_DIR / 'qa_pairs'}")
    print("="*60 + "\n")

# --- Main Execution Orchestrator ---

def main():
    """Main function to run the enhanced processing pipeline."""
    # Display the current state before we begin
    show_enhanced_resume_info()

    # Confirm to start
    start = input("Do you want to start the enhanced processing pipeline? (y/n): ")
    if start.lower() != 'y':
        print("🚫 Processing cancelled by user.")
        return

    # Create the main processor instance
    processor = EnhancedTextbookProcessor()
    
    # Run the full pipeline
    processor.process_all_files()

if __name__ == "__main__":
    # This block provides a user-friendly command-line interface.
    
    # --- Initial Setup ---
    # Attempt to configure the Gemini API key right at the start.
    api_key_found = False
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")
        genai.configure(api_key=GOOGLE_API_KEY)
        api_key_found = True
        logger.info("✅ Gemini API key loaded from Kaggle secrets.")
    except Exception:
        logger.warning("⚠️ Kaggle secrets not available, checking environment variables.")
        GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
        if GOOGLE_API_KEY:
            genai.configure(api_key=GOOGLE_API_KEY)
            api_key_found = True
            logger.info("✅ Gemini API key loaded from environment variable.")
        else:
            logger.error("❌ CRITICAL: GOOGLE_API_KEY not found in Kaggle secrets or environment.")
            logger.error("   The LLM-powered enrichment will FAIL. Please set the key.")

    if not api_key_found:
        print("\n" + "!"*80)
        print("! WARNING: GOOGLE API KEY IS NOT SET. LLM ENRICHMENT WILL NOT WORK. !")
        print("!"*80 + "\n")

    # --- Interactive Menu ---
    while True:
        print("\n--- Enhanced RAG Processing Menu ---")
        print("1. Run Full Processing Pipeline")
        print("2. Check Current Status")
        print("3. Reset All Processed Data (USE WITH CAUTION)")
        print("4. Exit")
        
        choice = input("Enter your choice (1-4): ")
        
        if choice == '1':
            try:
                main()
            except Exception as e:
                logger.critical(f"A fatal error occurred during the main pipeline execution: {e}", exc_info=True)
                print(f"❌ PIPELINE FAILED. Check the logs for details.")
            break # Exit menu after a run attempt
        elif choice == '2':
            show_enhanced_resume_info()
        elif choice == '3':
            reset_processing()
        elif choice == '4':
            print("👋 Exiting.")
            break
        else:
            print("Invalid choice. Please enter a number between 1 and 4.")

2025-08-07 15:56:38,281 - MainProcess - WARNING - ⚠️ Kaggle secrets not available, checking environment variables.
2025-08-07 15:56:38,281 - MainProcess - INFO - ✅ Gemini API key loaded from environment variable.



--- Enhanced RAG Processing Menu ---
1. Run Full Processing Pipeline
2. Check Current Status
3. Reset All Processed Data (USE WITH CAUTION)
4. Exit

🚀 ENHANCED PIPELINE - RESUME & CONFIGURATION INFO
📊 No processing metadata found. Looks like a fresh run.

   --- CURRENT CONFIGURATION ---
   Embedding Model: all-MiniLM-L6-v2
   LLM Model: gemini-1.5-flash
   Relevance Thresholds: Min=0.6, Core=0.8
   Chunk Size (Chars): Min=200, Max=800
   Max Workers: 4

   --- OUTPUT DIRECTORIES ---
   Core Chunks: Exists -> kaggle/working/enriched_chunks/core
   Longtail Chunks: Exists -> kaggle/working/enriched_chunks/longtail
   Q&A Pairs: Exists -> kaggle/working/fine_tuning_data/qa_pairs



2025-08-07 15:56:40,628 - MainProcess - INFO - Use pytorch device_name: cuda
2025-08-07 15:56:40,628 - MainProcess - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2025-08-07 15:56:44,118 - MainProcess - WARNING - spaCy model not found. Using NLTK fallback.
2025-08-07 15:56:44,119 - MainProcess - INFO - 🚀 ENHANCED PROCESSING PIPELINE
2025-08-07 15:56:44,119 - MainProcess - INFO - 📚 Found 219 PDF files
2025-08-07 15:56:44,119 - MainProcess - INFO - 🧩 Semantic chunking enabled: True
2025-08-07 15:56:44,120 - MainProcess - INFO - ✨ LLM enrichment enabled: True
2025-08-07 15:56:44,120 - MainProcess - INFO - 📊 Relevance threshold: 0.6
2025-08-07 15:56:44,120 - MainProcess - INFO - 🎯 High relevance threshold: 0.8
2025-08-07 15:56:44,120 - MainProcess - INFO - 
📖 Processing 1/219: 1-case-Post-kala-azar Dermal Leishmaniasis with Mucosal Involvement_ An Unusual Case Presentation including Successful Treatment with Miltefosine.pdf
2025-08-07 15:56:44,120 - MainProcess - INFO - 🚀 St

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-08-07 15:56:44,522 - MainProcess - INFO - ✨ Enriching chunks with LLM...
2025-08-07 15:56:44,522 - MainProcess - INFO - Enriching chunk 1/14: 1-case-Post-kala-azar Dermal Leishmaniasis with Mucosal Involvement_ An Unusual Case Presentation including Successful Treatment with Miltefosine_s5_c0
2025-08-07 15:56:44,522 - MainProcess - INFO - Enriching chunk 1/14: 1-case-Post-kala-azar Dermal Leishmaniasis with Mucosal Involvement_ An Unusual Case Presentation including Successful Treatment with Miltefosine_s5_c0
2025-08-07 15:56:50,489 - MainProcess - INFO - Enriching chunk 2/14: 1-case-Post-kala-azar Dermal Leishmaniasis with Mucosal Involvement_ An Unusual Case Presentation including Successful Treatment with Miltefosine_s8_c0
2025-08-07 15:56:56,104 - MainProcess - INFO - Enriching chunk 3/14: 1-case-Post-kala-azar Dermal Leishmaniasis with Mucosal Involvement_ An Unusual Case Presentation including Successful Treatment with Miltefosine_s10_c0
